# Staging-yard operations

A worked example of the scenario you described:

1. Several trains are parked in staging tracks (NW Track 1–6, SW Track 1–4).
2. For each train you want to *launch* it — align the staging turnouts to its track, pull it out onto the layout, let it run.
3. At some later point you want to *send it home* — re-align the staging turnouts, drive it back, stop in its track.
4. You may want two or three trains doing this simultaneously and on independent schedules.

The pattern below uses three pyjmri building blocks:

- **Routes** (`layout.routes["NW Track 3"].activate()`) to set every staging turnout to the saved positions for one specific track in one call.
- **Throttles** held open across cells so each train can be commanded independently.
- **`asyncio.Event`** as the “come home now” signal — one event per train, lets you tell train A to return without disturbing trains B and C.

**Target**: `192.168.1.159:12080`. Commands sent here actually move trains. If you are testing on the simulator, point the URL at it instead — the orchestration works either way, but the simulator has no virtual locos so no motion will be visible.

## 1. Imports, logging, connect

In [ ]:
import asyncio
import logging
from dataclasses import dataclass, field

from pyjmri import Client  # pyright: ignore[reportMissingImports]

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
    filename="pyjmri.log",
    force=True,
)

In [ ]:
jmri = await Client("192.168.1.159:12080").__aenter__()
layout = await jmri.discover()
print(f"discovered: {len(layout.turnouts)} turnouts, {len(layout.sensors)} sensors, {len(layout.routes)} routes, {len(layout.blocks)} blocks")

## 2. Find the routes and sensors you will reference

The basement panel already has the routes you need — one per staging track plus `*Staging Close` to put every staging turnout into a neutral state. Listing them once at the top of the session is the easiest way to confirm names (the dataclass below references them by `user_name`).

In [ ]:
for r in layout.routes.values():
    label = r.user_name or r.name
    if "Track" in label or "Staging" in label or "Bypass" in label:
        print(f"{r.name:10}  {label}")

In [ ]:
for s in layout.sensors.values():
    label = s.user_name or ""
    if "Staging" in label or "Block" in label:
        print(f"{s.name:8}  {label}")

## 3. Describe each train as data

Everything you need to launch and return one train is captured in this dataclass. Adding a second or third train is just appending another `Train(...)` to the list — no extra orchestration code.

Fields:

- `dcc`, `long` — the locomotive's DCC address.
- `staging_route` — user-name of the JMRI route that aligns the staging ladder onto this train's parking track (e.g. `"NW Track 3"`).
- `clear_route` — user-name of the route that puts the ladder back to a safe / neutral state once the train is out of the yard (e.g. `"NW Staging Close"`). Optional — leave as `None` if you don't want to reset between depart and return.
- `clear_sensor` — a block sensor that goes `ACTIVE` once the train has fully cleared the staging throat. Used to decide the train is safely on the main and we can stop watching the depart phase.
- `approach_seconds` — how long to crawl at `approach_speed` before stopping in the staging track. **This is the timing-based stand-in for a spot detector.** When you wire spot detectors in later, add an `arrival_sensor` field to the dataclass and swap the `asyncio.sleep(...)` in §4 for `await arrival_sensor.wait_active()` — the rest of the orchestration stays the same.
- `cruise_speed` / `approach_speed` — normal running speed vs. the slower speed used for the final crawl into staging.
- `come_home` — an `asyncio.Event` created automatically. **You set this from another cell when you want this specific train to return** — see §6.

In [ ]:
@dataclass
class Train:
    name: str
    dcc: int
    long: bool
    staging_route: str
    clear_sensor: str
    clear_route: str | None = None
    cruise_speed: float = 0.35
    approach_speed: float = 0.12
    approach_seconds: float = 15.0
    come_home: asyncio.Event = field(default_factory=asyncio.Event)

In [ ]:
# Edit these to match the trains you actually want to run. The route
# and sensor names must match the user-names printed in §2 exactly.
# `approach_seconds` is calibrated by trial: time how long the loco
# takes to drift from the clear_sensor back into its parking spot at
# `approach_speed`, then plug that number in (round down a little so
# you stop short of the bumper rather than past it).
trains = [
    Train(
        name="5327",
        dcc=5327, long=True,
        staging_route="NW Track 3",
        clear_route="NW Staging Close",
        clear_sensor="Block 1",
        approach_seconds=18.0,
    ),
    Train(
        name="1029",
        dcc=1029, long=True,
        staging_route="SW Track 2",
        clear_route="SW Staging Close",
        clear_sensor="Block 7",
        approach_seconds=15.0,
    ),
]
{t.name: t for t in trains}

## 4. One coroutine that runs one train end-to-end

`run_train` is the whole life cycle for a single locomotive:

1. Activate the train's staging route so the ladder aligns to its track.
2. Acquire the throttle (`async with` — the throttle is released cleanly even if the task is cancelled).
3. **Depart**: drive forward at cruise speed, wait for the *clear* sensor to confirm the train is on the main, optionally re-align the yard.
4. **Run**: sit at cruise speed until the train's `come_home` event is set. The train keeps moving the whole time — `Event.wait()` only suspends this coroutine, the loco itself is open-loop on NCE.
5. **Return**: re-align the staging route to this train's track, drop to approach speed (still forward, since on the basement layout staging is entered nose-first), crawl for `approach_seconds`, then stop.

The return phase relies on **timing** rather than a spot detector — there is currently no sensor inside each staging track to tell us we have reached the bumper. Calibrate `approach_seconds` per train by trial. When spot detectors are installed, the only change is in this function: replace `asyncio.sleep(train.approach_seconds)` with `await arrival_sensor.wait_active()`.

Everything is wrapped in `try / finally` so an emergency `task.cancel()` still zeroes the throttle before releasing it.

In [ ]:
async def run_train(layout, train: Train) -> None:
    clear_sensor = layout.sensors[train.clear_sensor]
    staging_route = layout.routes[train.staging_route]
    clear_route = layout.routes[train.clear_route] if train.clear_route else None

    print(f"[{train.name}] aligning {train.staging_route!r}")
    await staging_route.activate()

    async with layout.throttle(train.dcc, long=train.long) as t:
        try:
            print(f"[{train.name}] departing")
            await t.set_speed(train.cruise_speed, forward=True)
            await clear_sensor.wait_active()
            print(f"[{train.name}] on the main")

            if clear_route is not None:
                await clear_route.activate()

            await train.come_home.wait()
            print(f"[{train.name}] return commanded")

            await staging_route.activate()
            await t.set_speed(train.approach_speed, forward=True)
            await asyncio.sleep(train.approach_seconds)
            print(f"[{train.name}] arrived (timed)")
        finally:
            await t.set_speed(0.0, forward=True)

## 5. Launch trains as background tasks

Wrapping each `run_train(...)` in `asyncio.create_task(...)` lets the cell return immediately. The trains keep running on the event loop, and the notebook stays interactive — you can issue ad-hoc throttle commands, watch sensors, or open §6 to send a train home.

Each task is bound to a key in the `tasks` dict by train name, so later cells can reach in and cancel just one without touching the others. Staggering the launches with a short `asyncio.sleep` between them avoids the two trains fighting over the same staging ladder at the same instant — each gets to activate its route, pull out, and clear before the next one tries.

In [ ]:
tasks: dict[str, asyncio.Task] = {}

for train in trains:
    tasks[train.name] = asyncio.create_task(run_train(layout, train))
    await asyncio.sleep(20)  # let this one clear staging before the next pulls out

print(f"launched {len(tasks)} trains: {list(tasks)}")

## 6. Send a specific train home

Setting `train.come_home` on one `Train` instance unblocks exactly one `run_train` coroutine. The other trains keep cruising. You can do this from any cell, in any order, as often as you like — just edit the train name and re-run.

In [ ]:
trains_by_name = {t.name: t for t in trains}
trains_by_name["5327"].come_home.set()

In [ ]:
trains_by_name["1029"].come_home.set()

## 7. Wait for everything to finish

Once you have set every train's `come_home` event, this cell blocks until each `run_train` returns (i.e. every train has parked). `return_exceptions=True` keeps one train's failure from masking the others.

In [ ]:
results = await asyncio.gather(*tasks.values(), return_exceptions=True)
for name, result in zip(tasks, results):
    print(f"{name}: {result!r}")

## 8. Emergency stop — cancel one or all trains

Cancelling the task raises `CancelledError` inside `run_train`. The `finally` clause sets the loco to 0 and the `async with layout.throttle(...)` block then releases the throttle. The train will *not* finish its return sequence — it just stops wherever it is.

In [ ]:
# Cancel one train by name.
# tasks["5327"].cancel()
# await asyncio.gather(tasks["5327"], return_exceptions=True)

In [ ]:
# Cancel every running train.
# for task in tasks.values():
#     if not task.done():
#         task.cancel()
# await asyncio.gather(*tasks.values(), return_exceptions=True)

## 9. A realistic per-train script

The pattern in §3–§8 is the simplest thing that runs multiple trains. Real locomotives need more than a single departure speed and a timed crawl back into staging:

- **Startup and shutdown rituals.** Headlight (F0), prime mover sound (F9), a tiny brake-test nudge, a horn salute (F2), all paced by `asyncio.sleep` so the decoder's sound files have time to play. Shutdown is the reverse, plus a re-toggle of F9 to trigger the decoder's shutdown sound.
- **Function calls** for each of those. pyjmri exposes them as `await t.set_function(n, on)` where `n` is any function number 0–28.
- **A slowdown profile** instead of one approach speed: drop speed at sensor A, drop again at sensor B, and stop on the trailing edge of B.
- **A separate return route** — NW staging is exited via `NW Track <n>` but re-entered in reverse via `NW Bypass`, so depart and return are not symmetric.
- **Edge detection** (`wait_active` followed by `wait_inactive`) for "the loco has fully passed this sensor" rather than just "reached it."

Per-loco sequences are inherently hand-tuned — every decoder has its own sound mapping, every consist has different inertia — so the cleanest pattern is to write one async function per train rather than over-parameterising a dataclass. The cells below define three small helpers (`horn`, `wait_edge`, `slow_through`) and then one full per-train function, `run_8997`, mirroring `jython/MikeStartNW8997.py`.

### 9.1 Helpers

In [ ]:
async def horn(t, *, duration: float = 1.2, pause: float = 0.5, times: int = 3) -> None:
    """Toggle F2 to blow the horn `times` times."""
    for _ in range(times):
        await t.set_function(2, True)
        await asyncio.sleep(duration)
        await t.set_function(2, False)
        await asyncio.sleep(pause)


async def wait_edge(sensor) -> None:
    """Wait for the sensor to go ACTIVE, then back to INACTIVE.

    The trailing edge means the loco has fully passed the detection
    block — used to know the train has cleared a turnout, not just
    reached it.
    """
    await sensor.wait_active()
    await sensor.wait_inactive()


async def slow_through(layout, t, schedule: list[tuple[str, float]]) -> None:
    """At each (sensor_user_name, speed) checkpoint, wait for the sensor
    to go ACTIVE, then drop the throttle to the listed speed."""
    for sensor_name, speed in schedule:
        await layout.sensors[sensor_name].wait_active()
        await t.set_speed(speed, forward=True)

### 9.2 One train, written out

`run_8997` is the whole life cycle for SP 8997 starting from NW Staging Track 6 and returning into the same track via the NW Bypass. It mirrors `jython/MikeStartNW8997.py` but uses pyjmri primitives directly. The structure is the same five phases as `run_train` in §4 — align route, depart, cruise, return, park — with the addition of sound/light rituals at the start and end.

Each phase is hand-tuned for this specific decoder, route, and the timings of this particular train. Writing a similar function for a different loco is a matter of changing the route names, sensor names, F-numbers, and timings; the shape stays the same. The `come_home: asyncio.Event` argument is the only thing that ties it to the multi-train pattern from §5 — pass an `asyncio.Event` per train when you launch, set it from another cell to bring that one train home.

In [ ]:
async def run_8997(layout, come_home: asyncio.Event) -> None:
    nw_track_6 = layout.routes["NW Track 6"]
    nw_bypass = layout.routes["NW Bypass"]
    nw_close = layout.routes["NW Staging Close"]
    straight_through = layout.routes["Straight Through"]
    throat = layout.sensors["West / SW"]
    lap_start = layout.sensors["North Zone 1"]

    print("[8997] aligning NW Track 6")
    await nw_track_6.activate()

    async with layout.throttle(8997, long=True) as t:
        try:
            # Startup — lights and sounds before any motion.
            await t.set_function(0, True)            # headlight
            await t.set_speed(0.01, forward=True)    # brake test nudge
            await asyncio.sleep(0.2)
            await t.set_speed(0.0, forward=True)
            await t.set_function(9, True)            # prime mover
            await asyncio.sleep(20)                  # let startup sound play
            await horn(t, duration=1.5, pause=1.2, times=3)

            # Depart at a crawl, wait for the loco to fully cross the
            # throat sensor, then close the staging ladder behind it.
            print("[8997] departing")
            await t.set_speed(0.10, forward=True)
            await wait_edge(throat)
            await nw_close.activate()

            # Open up to cruise and salute the start of the lap.
            await straight_through.activate()
            await t.set_speed(0.35, forward=True)
            await lap_start.wait_active()
            await horn(t, duration=1.2, pause=0.8, times=2)

            # Cruise — the loco keeps moving while we wait. NCE doesn't
            # need a continuous packet stream to hold speed.
            await come_home.wait()
            print("[8997] return commanded")

            # Slowdown profile on the next approach to staging.
            await slow_through(layout, t, [
                ("North Zone 9", 0.20),
                ("West / SW",    0.15),
            ])
            await throat.wait_inactive()
            await t.set_speed(0.0, forward=True)
            await asyncio.sleep(10)

            # Reverse into staging via the bypass route.
            await nw_bypass.activate()
            await t.set_function(1, True)            # bell on for reverse move
            await t.set_speed(0.15, forward=False)
            await wait_edge(throat)
            await t.set_speed(0.10, forward=False)
            await asyncio.sleep(28)                  # timed crawl into the spot
            await t.set_speed(0.0, forward=False)

            # Shutdown — bell and headlight off, toggle F9 to trigger
            # the decoder's shutdown sound, then wait for it to finish.
            await t.set_function(1, False)
            await t.set_function(0, False)
            await t.set_function(9, False)
            await t.set_function(9, True)
            await asyncio.sleep(30)
            await nw_close.activate()
            print("[8997] parked")
        finally:
            await t.set_speed(0.0, forward=True)

In [ ]:
come_home_8997 = asyncio.Event()
task_8997 = asyncio.create_task(run_8997(layout, come_home_8997))
print("8997 launched — startup ritual runs for ~25 s before the loco moves")

In [ ]:
come_home_8997.set()

In [ ]:
await task_8997

## 10. Shutdown

Close the client when you are done. Throttles are released by their `async with` blocks inside `run_train` and `run_8997`, so there is nothing else to clean up at this layer.

In [ ]:
await jmri.__aexit__(None, None, None)
print("client closed")

## Notes and gotchas

- **`clear_sensor` must really exist** — the user-names in §3 are placeholders. Replace them with sensors you actually have wired (a block detector at the staging throat is the natural choice).
- **Timed approach, not detected arrival** — there are no spot detectors in staging today, so the final stop is `asyncio.sleep(approach_seconds)`. Calibrate per train: run the train back at `approach_speed`, count seconds from `clear_sensor` ACTIVE to the nose hitting the bumper, subtract a safety margin, and store that as `approach_seconds`. Be aware that decoder momentum, traction-tire wear, and how dirty the rails are will all drift this number over time — re-calibrate occasionally. When you add spot detectors, the change is local to §4 (swap the sleep for `await arrival_sensor.wait_active()`).
- **If two trains share a staging ladder**, do not launch them in parallel without a delay — they will race for the turnouts. The 20-second `asyncio.sleep` in §5 is a crude lockout; if you want to be rigorous, gate each train's depart phase on the previous one's `clear_sensor` going `INACTIVE` again (i.e. the lead train has fully left the throat).
- **NCE is open-loop** — a route activation is a *commanded* outcome, not an observed one. If you suspect a turnout did not move, drive the layout manually or look at JMRI's panel; pyjmri cannot tell you what the points are actually doing.
- **Reverse-into-staging layouts**: this notebook assumes trains enter staging nose-first. If yours back in, flip `forward=True` to `forward=False` in the return phase of `run_train`.
- **`come_home` is one-shot**: once set, the event stays set. To reuse a `Train` for a second lap, replace its event (`train.come_home = asyncio.Event()`) before relaunching.

## 11. Operations subsystem (read-only)

Read-only discovery of JMRI's **Operations** module — locations, trains, cars, and engines — added in pyjmri v1.1 (`discover_operations()`). This is the *operationally-active* picture (what is deployed on the layout right now), distinct from both the `Layout` discovered above and the DecoderPro roster.

**Target**: `192.168.1.159:12080` — the physical layout. Every cell here only *reads* Operations data; nothing moves a train or changes layout state, so it is safe to run during a live operating session.

This section opens **its own** client (`ops_jmri`) so it can be run on its own — the staging-yard session above closes `jmri` in §10. Run §11.1 first, then any cell below, and §11.8 to close.

### 11.1 Connect and discover Operations

In [ ]:
from pyjmri import Client  # pyright: ignore[reportMissingImports]

# Opens a dedicated client so this section is independent of the staging-yard
# session above (which closes `jmri` in §10).
ops_jmri = await Client("192.168.1.159:12080").__aenter__()
ops = await ops_jmri.discover_operations()
print(
    f"discovered operations: {len(ops.locations)} locations, "
    f"{len(ops.trains)} trains, {len(ops.cars)} cars, {len(ops.engines)} engines"
)


def placement(p):
    """Render a car/engine location or destination (or '—' when unplaced)."""
    if p is None:
        return "—"
    where = p.user_name or p.name
    return f"{where} / {p.track.user_name or p.track.name}" if p.track else where

### 11.2 Locations and their tracks

In [ ]:
for loc in ops.locations.values():
    tracks = ", ".join((t.user_name or t.name) for t in loc.tracks) or "—"
    print(f"{loc.name:4} {loc.user_name or '—':22} length={loc.length:>5}  tracks: {tracks}")

### 11.3 Trains — where each one is

Each train carries its assigned route, the location it is currently at (`None`/`—` before it departs), JMRI's freeform status string, and its consist (engines + cars + ordered route stops).

In [ ]:
for train in ops.trains.values():
    print(train.user_name or train.name)
    print(f"    route:   {train.route or '—'}")
    print(f"    at:      {train.current_location or '—'}")
    print(f"    status:  {train.status!r}  (lead engine: {train.lead_engine or '—'})")
    print(
        f"    consist: {len(train.engines)} engine(s), "
        f"{len(train.cars)} car(s), {len(train.route_stops)} route stop(s)"
    )

### 11.4 Where is every car

This is PRD Journey 5 — the same report as `examples/operations_report.py`. Each car shows its current location/track, the train it is assigned to, and its destination.

In [ ]:
for car in ops.cars.values():
    print(
        f"{car.name:10} {car.car_type:14} "
        f"at {placement(car.location):30} "
        f"on {car.train or '—':14} "
        f"-> {placement(car.destination)}"
    )

### 11.5 Engines — deployed vs idle

Operations engines are the subset actually on the layout, not the full DecoderPro roster. Here you can see which are assigned to a train and which are idle.

In [ ]:
for eng in ops.engines.values():
    deployment = f"assigned to {eng.train}" if eng.train else "idle (not on a train)"
    print(
        f"{eng.name:10} {eng.engine_type:8} model={eng.model or '—':10} "
        f"at {placement(eng.location):30} — {deployment}"
    )

### 11.6 Look entities up by name

Locations and trains resolve by **both** user name and system id to the same object; cars and engines have no user name and resolve by road+number. A missing key raises `LayoutEntityNotFound`.

In [ ]:
named = next((loc for loc in ops.locations.values() if loc.user_name), None)
if named is not None:
    by_user = ops.locations[named.user_name]
    by_system = ops.locations[named.name]
    print(f"{named.user_name!r} and system id {named.name!r} -> same object: {by_user is by_system}")

car = next(iter(ops.cars.values()), None)
if car is not None:
    print(f"car {car.name!r}: user_name={car.user_name!r}; looked up by road+number -> {ops.cars[car.name].name}")

### 11.7 The read-only boundary

Operations is read-only in this release: there is no build-train / move-assign-car / generate-manifest surface (those are deferred to a future command increment). The container exposes only its four collections, and the entity snapshots are frozen.

In [ ]:
methods = [m for m in dir(ops) if not m.startswith("_") and callable(getattr(ops, m))]
print("Operations container methods:", methods or "(none — only the four read-only collections)")

car = next(iter(ops.cars.values()), None)
if car is not None:
    try:
        car.train = "REASSIGNED"  # type: ignore[misc]  # frozen snapshot -> raises
    except Exception as exc:
        print(f"mutation blocked: {type(exc).__name__}: {exc}")

### 11.8 Close the Operations client

In [ ]:
await ops_jmri.__aexit__(None, None, None)
print("operations client closed")

## 12. Adapting this notebook into a standalone script

The notebook works because Jupyter runs an event loop *for* you and lets every cell `await` at the top level, keeping one `jmri` connection — and any background train tasks — alive **between** cells. A `.py` file has none of that: there is no running loop until you start one, and nothing survives past the function that created it. Porting is mostly about **rehoming every `await` inside a single `async def main()`** and letting `async with` manage lifetimes. The helpers move into their own files in the same directory and are imported normally.

### Suggested file layout

```
staging/
├── opsreport.py     # pure formatting + the read-only Operations report (§11)
├── trains.py        # Train dataclass + async run_train() + async horn() (§3, §4, §9)
└── run_session.py   # entry point: async def main() + asyncio.run(main())
```

The rule for helper modules: **they may define `async def` coroutines, but they must never call `asyncio.run()` or use top-level `await`.** They are *awaited by* the caller; they never start a loop themselves. Only `run_session.py` starts the loop, exactly once.

`opsreport.py` — synchronous; it takes an already-discovered `Operations`, so there is no `await` in it at all:

```python
from pyjmri import Operations, Placement


def placement(p: Placement | None) -> str:
    if p is None:
        return "—"
    where = p.user_name or p.name
    return f"{where} / {p.track.user_name or p.track.name}" if p.track else where


def print_report(ops: Operations) -> None:
    for car in ops.cars.values():
        print(f"{car.name:10} at {placement(car.location):30} on {car.train or '—'}")
```

`trains.py` — the coroutines from §3, §4, §9, unchanged except that they now live in a module:

```python
import asyncio
from dataclasses import dataclass, field


@dataclass
class Train:
    name: str
    track_route: str
    # ...the fields you already use in §3...
    come_home: asyncio.Event = field(default_factory=asyncio.Event)


async def run_train(layout, train: Train) -> None:
    ...  # exactly the body from §4
```

`field(default_factory=asyncio.Event)` is safe at import time (it does **not** build an Event until a `Train` is instantiated). But do not *construct* a `Train()` at module top level — build your trains **inside** `main()`, after the loop is running, so each `asyncio.Event` binds to the correct running loop.

### The actual loop changes (the part that bites)

**1. Every top-level `await` moves inside `async def main()`, and you call `asyncio.run(main())` exactly once.** You cannot `await` at module level, and you cannot call `asyncio.run()` from inside a coroutine or more than once per process.

Notebook (separate cells, §1 and §10):

```python
jmri = await Client("192.168.1.159:12080").__aenter__()   # one cell
layout = await jmri.discover()                            # another cell
...
await jmri.__aexit__(None, None, None)                    # a later cell
```

Script:

```python
async def main() -> None:
    async with Client("192.168.1.159:12080") as jmri:
        layout = await jmri.discover()
        ...
    # client is closed here automatically — even on exception or Ctrl-C


if __name__ == "__main__":
    asyncio.run(main())
```

The manual `__aenter__` / `__aexit__` split only existed so the connection could straddle cells. In a script, `async with Client(...) as jmri:` is the whole story — open at the top of `main()`, guaranteed close at the bottom.

**2. Background tasks must be awaited before `main()` returns.** In the notebook, `asyncio.create_task(run_train(...))` (§5) keeps running because the kernel's loop outlives the cell. In a script, the instant `main()` returns, `asyncio.run` cancels anything still pending — your trains would simply stop mid-run. Collect the tasks and wait for them. Prefer a `TaskGroup` (3.11+), which also cancels siblings cleanly if one raises:

```python
async with asyncio.TaskGroup() as tg:
    for train in trains:
        tg.create_task(run_train(layout, train))
# falls through only once every train has finished
```

(The notebook's `await asyncio.gather(*tasks.values(), return_exceptions=True)` in §7 does the same job; `TaskGroup` is the structured-concurrency upgrade and is what `pyjmri` uses internally.)

**3. Replace the "run a later cell" triggers with a real signal.** The come-home pattern works in the notebook because *you* run §6 (`trains_by_name["1029"].come_home.set()`) whenever you decide. A script has no later cell — choose a concrete trigger instead:

- **Fixed schedule** — a tiny driver coroutine: `await asyncio.sleep(600); train.come_home.set()`, launched as another task in the same `TaskGroup`.
- **Ctrl-C** — let `KeyboardInterrupt` propagate; `async with Client` still closes the connection. Catch it in `main()` to set every `come_home` and let the trains drive home before you exit.
- **Typed input, without freezing the loop** — `input()` is blocking, so run it off-thread and concurrently with the trains:

  ```python
  async def console(trains_by_name):
      while True:
          name = await asyncio.to_thread(input, "train to send home: ")
          trains_by_name[name].come_home.set()
  ```

  Add `console` as another `tg.create_task(...)`. A bare inline `input()` would block the **whole** event loop — every train would freeze while it waits for you to type.

**4. One coroutine, one loop, one `asyncio.run`.** Everything above runs on the single loop that `asyncio.run(main())` creates and tears down. Don't sprinkle `asyncio.run()` inside helpers, and don't spin up a second loop. If you ever feel you need `asyncio.run()` twice, that is the tell that two notebook cells should become two `await` statements inside the same `main()`.

### Minimal `run_session.py`

```python
import asyncio

from pyjmri import Client

from opsreport import print_report
from trains import Train, run_train


async def main() -> None:
    async with Client("192.168.1.159:12080") as jmri:
        layout = await jmri.discover()

        ops = await jmri.discover_operations()   # the read-only §11 report
        print_report(ops)

        trains = [Train(name="1029", track_route="NW Track 3")]  # build INSIDE main()
        async with asyncio.TaskGroup() as tg:
            for train in trains:
                tg.create_task(run_train(layout, train))


if __name__ == "__main__":
    asyncio.run(main())
```

Run it from this directory so the same-directory `opsreport` / `trains` imports resolve:

```bash
uv run --no-sync python run_session.py
```

(or `python -m run_session` if you turn the folder into a package). Ctrl-C is a clean exit: the `async with Client(...)` block closes the connection on the way out, and any throttles opened inside `run_train` are released by their own `async with` blocks.

### Checklist

- [ ] No `await` outside an `async def`; no `asyncio.run()` outside the `if __name__ == "__main__":` guard.
- [ ] `Client` used as `async with`, not manual `__aenter__` / `__aexit__`.
- [ ] Every `create_task` is awaited (via `TaskGroup` or `gather`) before `main()` returns.
- [ ] `Train` instances (and their `asyncio.Event`s) are built **inside** `main()`, not at import time.
- [ ] The come-home trigger is a real source (timer, signal, or off-thread `input`), not a cell you re-run.
- [ ] Helper modules contain coroutines but never start a loop.